# Week 7 Maintainer's Copilot — Pandas ML Dataset + Classical Baseline

Repository:

```text
pandas-dev/pandas
```

This notebook follows the Week 7 project plan for the ML dataset/classification part:

- use closed issues from one open-source repo
- map maintainer labels into `bug`, `feature`, `docs`, `question`
- create a time-aware split
- check dataset quality before continuing
- train the first required baseline: TF-IDF + Logistic Regression


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib requests tqdm joblib

In [ ]:
import os
import re
import json
import time
from pathlib import Path
from datetime import datetime

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


In [ ]:
OWNER = "pandas-dev"
REPO = "pandas"

PROJECT_LABELS = ["bug", "feature", "docs", "question"]

MAX_ISSUES_PER_LABEL = 300

RAW_DATA_PATH = Path("issues_raw_pandas_label_fetch.jsonl")
PROCESSED_DATA_PATH = Path("issues_processed_pandas_label_fetch.csv")
DATASET_SUMMARY_PATH = Path("dataset_summary_pandas.json")


## Optional GitHub token

If GitHub rate-limits you, uncomment the `getpass` lines and paste a GitHub token.


In [ ]:
# import getpass
# GITHUB_TOKEN = getpass.getpass("Enter GitHub token: ")

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

headers = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

if GITHUB_TOKEN:
    headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"

print("Token configured:", bool(GITHUB_TOKEN))


## Label groups

Pandas labels are cleaner for this project:

- `Bug` → `bug`
- `Enhancement` → `feature`
- `Docs` → `docs`
- `Usage Question` → `question`


In [ ]:
LABEL_GROUPS = {
    "bug": ["Bug"],
    "feature": ["Enhancement"],
    "docs": ["Docs"],
    "question": ["Usage Question"],
}

LABEL_MAPPING = {
    "bug": "bug",
    "enhancement": "feature",
    "docs": "docs",
    "documentation": "docs",
    "doc": "docs",
    "usage question": "question",
    "question": "question",
}


In [ ]:
def fetch_closed_issues_by_label(owner: str, repo: str, github_label: str, max_issues: int = 300) -> list[dict]:
    """Fetch closed GitHub issues for one specific maintainer label."""
    issues = []
    page = 1
    per_page = 100

    while len(issues) < max_issues:
        url = f"https://api.github.com/repos/{owner}/{repo}/issues"
        params = {
            "state": "closed",
            "labels": github_label,
            "per_page": per_page,
            "page": page,
            "sort": "created",
            "direction": "asc",
        }

        response = requests.get(url, headers=headers, params=params)

        if response.status_code != 200:
            print(f"GitHub API error for label {github_label}:", response.status_code)
            print(response.text[:700])
            break

        batch = response.json()

        if not batch:
            break

        for item in batch:
            # GitHub's issues endpoint returns pull requests too.
            # Exclude pull requests because the project dataset is GitHub issues.
            if "pull_request" in item:
                continue

            issues.append(item)

            if len(issues) >= max_issues:
                break

        print(f"Label {github_label!r} | page {page} | total: {len(issues)}")

        page += 1
        time.sleep(0.25)

    return issues


In [ ]:
all_issues = []
seen_issue_numbers = set()
fetch_report_rows = []

for target_label, github_labels in LABEL_GROUPS.items():
    for github_label in github_labels:
        fetched = fetch_closed_issues_by_label(
            OWNER,
            REPO,
            github_label,
            max_issues=MAX_ISSUES_PER_LABEL,
        )

        fetch_report_rows.append({
            "target_label": target_label,
            "github_label": github_label,
            "fetched_count": len(fetched),
        })

        for issue in fetched:
            issue_number = issue["number"]

            if issue_number in seen_issue_numbers:
                continue

            issue["target_label_from_fetch"] = target_label
            all_issues.append(issue)
            seen_issue_numbers.add(issue_number)

issues = all_issues

fetch_report = pd.DataFrame(fetch_report_rows)
display(fetch_report)
print("Total unique fetched issues:", len(issues))


In [ ]:
with RAW_DATA_PATH.open("w", encoding="utf-8") as f:
    for issue in issues:
        f.write(json.dumps(issue, ensure_ascii=False) + "\n")

print(f"Saved raw issues to: {RAW_DATA_PATH}")


## Inspect labels from fetched issues

In [ ]:
all_labels = []

for issue in issues:
    labels = [label["name"] for label in issue.get("labels", [])]
    all_labels.extend(labels)

label_counts = pd.Series(all_labels).value_counts()
label_counts.head(100)


## Build processed dataset

In [ ]:
def map_issue_label(github_labels: list[str], fallback_label: str | None = None) -> str | None:
    """Map GitHub maintainer labels into the four project labels."""
    normalized_labels = [label.lower().strip() for label in github_labels]
    matched_project_labels = []

    for label in normalized_labels:
        if label in LABEL_MAPPING:
            matched_project_labels.append(LABEL_MAPPING[label])

        # Flexible safety checks for label variants.
        if "bug" in label:
            matched_project_labels.append("bug")

        if "doc" in label:
            matched_project_labels.append("docs")

        if "enhancement" in label or "feature" in label:
            matched_project_labels.append("feature")

        if "question" in label or "usage" in label:
            matched_project_labels.append("question")

    # Since we fetched by clean label group, this fallback is acceptable.
    # It records which target label the GitHub label was collected for.
    if fallback_label:
        matched_project_labels.append(fallback_label)

    # Priority prevents weak/general labels from overriding stronger issue type labels.
    priority = ["bug", "docs", "feature", "question"]

    for project_label in priority:
        if project_label in matched_project_labels:
            return project_label

    return None


In [ ]:
rows = []

for issue in issues:
    github_labels = [label["name"] for label in issue.get("labels", [])]
    mapped_label = map_issue_label(
        github_labels,
        fallback_label=issue.get("target_label_from_fetch"),
    )

    if mapped_label is None:
        continue

    title = issue.get("title") or ""
    body = issue.get("body") or ""
    text = f"Title: {title}\n\nBody: {body}"

    rows.append({
        "github_issue_id": issue["number"],
        "repo": f"{OWNER}/{REPO}",
        "title": title,
        "body": body,
        "text": text,
        "github_labels": github_labels,
        "mapped_label": mapped_label,
        "target_label_from_fetch": issue.get("target_label_from_fetch"),
        "state": issue.get("state"),
        "created_at": issue.get("created_at"),
        "closed_at": issue.get("closed_at"),
        "url": issue.get("html_url"),
    })

df = pd.DataFrame(rows)

print("Processed dataset size:", len(df))
display(df.head())
print(df["mapped_label"].value_counts())


## Clean issue text

In [ ]:
def clean_issue_text(text: str) -> str:
    """Clean issue text while preserving useful technical terms."""
    if not isinstance(text, str):
        return ""

    text = text.replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = text.strip()

    max_chars = 5000
    if len(text) > max_chars:
        text = text[:max_chars]

    return text


df["clean_text"] = df["text"].apply(clean_issue_text)
df[["github_issue_id", "mapped_label", "clean_text"]].head()


## Class-wise time-aware split

This keeps the time ordering inside each class.

Reason: the project requires a time-aware split, but we also need every class to appear in train/validation/test for a usable four-class classifier.


In [ ]:
df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
df = df.dropna(subset=["created_at"]).copy()

split_parts = []

for label, group in df.groupby("mapped_label"):
    group = group.sort_values("created_at").reset_index(drop=True)
    n = len(group)

    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    group.loc[:train_end - 1, "split"] = "train"
    group.loc[train_end:val_end - 1, "split"] = "val"
    group.loc[val_end:, "split"] = "test"

    split_parts.append(group)

df = pd.concat(split_parts, ignore_index=True)
df = df.sort_values(["split", "created_at"]).reset_index(drop=True)

print("Overall class distribution:")
display(df["mapped_label"].value_counts())

print("Split distribution:")
display(pd.crosstab(df["split"], df["mapped_label"]))


## Dataset quality gate

Do not continue to transformer training unless all four classes exist in train, validation, and test.


In [ ]:
split_table = pd.crosstab(df["split"], df["mapped_label"])

required_splits = {"train", "val", "test"}
required_labels = set(PROJECT_LABELS)

problems = []

for split_name in required_splits:
    if split_name not in split_table.index:
        problems.append(f"Missing split: {split_name}")
        continue

    for label in required_labels:
        if label not in split_table.columns:
            problems.append(f"Missing label column: {label}")
            continue

        count = int(split_table.loc[split_name, label])
        if count == 0:
            problems.append(f"Class {label!r} has 0 examples in split {split_name!r}")

if problems:
    print("DATASET QUALITY: NOT READY")
    for problem in problems:
        print("-", problem)
else:
    print("DATASET QUALITY: READY ENOUGH TO TRAIN BASELINE")

split_table


In [ ]:
df.to_csv(PROCESSED_DATA_PATH, index=False)

dataset_summary = {
    "repo": f"{OWNER}/{REPO}",
    "processed_dataset_size": int(len(df)),
    "class_distribution": df["mapped_label"].value_counts().to_dict(),
    "split_distribution": pd.crosstab(df["split"], df["mapped_label"]).to_dict(),
    "fetch_report": fetch_report.to_dict(orient="records"),
    "created_at": datetime.utcnow().isoformat() + "Z",
}

with DATASET_SUMMARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(dataset_summary, f, indent=2, ensure_ascii=False)

print(f"Saved processed dataset to: {PROCESSED_DATA_PATH}")
print(f"Saved dataset summary to: {DATASET_SUMMARY_PATH}")


## Classical baseline: TF-IDF + Logistic Regression

This is the first classifier required by the project plan.


In [ ]:
train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()
test_df = df[df["split"] == "test"].copy()

print("Train size:", len(train_df))
print("Val size:", len(val_df))
print("Test size:", len(test_df))

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
)

X_train = vectorizer.fit_transform(train_df["clean_text"])
X_val = vectorizer.transform(val_df["clean_text"])
X_test = vectorizer.transform(test_df["clean_text"])

y_train = train_df["mapped_label"]
y_val = val_df["mapped_label"]
y_test = test_df["mapped_label"]

clf = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42,
)

clf.fit(X_train, y_train)

val_preds = clf.predict(X_val)
test_preds = clf.predict(X_test)

val_acc = accuracy_score(y_val, val_preds)
val_macro_f1 = f1_score(y_val, val_preds, average="macro")

test_acc = accuracy_score(y_test, test_preds)
test_macro_f1 = f1_score(y_test, test_preds, average="macro")

print("Validation accuracy:", val_acc)
print("Validation macro-F1:", val_macro_f1)
print("Test accuracy:", test_acc)
print("Test macro-F1:", test_macro_f1)

print("\nClassification report:")
print(classification_report(y_test, test_preds, labels=PROJECT_LABELS, zero_division=0))


In [ ]:
cm = confusion_matrix(y_test, test_preds, labels=PROJECT_LABELS)

plt.figure(figsize=(7, 5))
plt.imshow(cm)
plt.title("Classical Baseline Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.xticks(range(len(PROJECT_LABELS)), PROJECT_LABELS, rotation=45)
plt.yticks(range(len(PROJECT_LABELS)), PROJECT_LABELS)
plt.colorbar()

for i in range(len(PROJECT_LABELS)):
    for j in range(len(PROJECT_LABELS)):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.savefig("classical_confusion_matrix_pandas.png", dpi=150)
plt.show()

print("Saved confusion matrix to: classical_confusion_matrix_pandas.png")


In [ ]:
metrics = {
    "repo": f"{OWNER}/{REPO}",
    "model": "TF-IDF + Logistic Regression",
    "validation_accuracy": float(val_acc),
    "validation_macro_f1": float(val_macro_f1),
    "test_accuracy": float(test_acc),
    "test_macro_f1": float(test_macro_f1),
    "labels": PROJECT_LABELS,
    "created_at": datetime.utcnow().isoformat() + "Z",
}

with open("classical_metrics_pandas.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

joblib.dump(
    {
        "vectorizer": vectorizer,
        "model": clf,
        "labels": PROJECT_LABELS,
        "repo": f"{OWNER}/{REPO}",
    },
    "classical_tfidf_logreg_pandas.joblib",
)

print("Saved metrics to: classical_metrics_pandas.json")
print("Saved model to: classical_tfidf_logreg_pandas.joblib")
metrics
